In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import entropy

MOVEMENT_DIR = Path(
    r"C:\Users\Sheel\OneDrive\Documents\Default Project\data\pads-parkinsons-disease-smartwatch-dataset-1.0.0\pads-parkinsons-disease-smartwatch-dataset-1.0.0\preprocessed\movement"
)

FS = 100.0

file_list = pd.read_csv(MOVEMENT_DIR.parent / "file_list.csv")

tasks = [
    "Relaxed1", "Relaxed2", "RelaxedTask1", "RelaxedTask2",
    "StretchHold", "HoldWeight", "DrinkGlas", "CrossArms",
    "TouchNose", "Entrainment1", "Entrainment2",
]
wrists = ["LeftWrist", "RightWrist"]
sensors = ["Accelerometer", "Gyroscope"]
axes = ["X", "Y", "Z"]
channels = [f"{t}_{w}_{s}_{a}" for t in tasks for w in wrists for s in sensors for a in axes]

print(len(channels), "channels")

132 channels


In [ ]:
bin_files = sorted(MOVEMENT_DIR.glob("*_ml.bin"))
ids = np.array([int(f.name.split("_")[0]) for f in bin_files])
movement = np.stack([np.fromfile(f, dtype=np.float32).reshape(len(channels), -1) for f in bin_files])
print("movement:", movement.shape, movement.dtype)
print("ids aligned with file_list:", np.array_equal(ids, file_list["id"].values))

movement: (469, 132, 976) float32
ids aligned with file_list: True


In [ ]:
channel_df = pd.DataFrame({
    "channel": channels,
    "task": [c.split("_")[0] for c in channels],
    "wrist": [c.split("_")[1] for c in channels],
    "sensor": [c.split("_")[2] for c in channels],
    "axis": [c.split("_")[3] for c in channels],
})

predictor_names = [
    "Primary Frequency Component", "Acceleration Norm", "Jerk",
    "Signal Change Variability", "Spectral Entropy",
]
        entropy(power[:, 1:], axis=1),
    ]
    for name, vec in zip(predictor_names, values):
    acc = movement[:, grp.index[grp["sensor"] == "Accelerometer"].tolist(), :]
        feature_names.append(f"{task}_{wrist}_{name}")

X = np.column_stack(stream_feats)
    conditions = ["Healthy", "Parkinson's", "Essential Tremor", "Atypical Parkinsonism", "Multiple Sclerosis"]
    keep = np.isin(file_list["condition"].to_numpy(), conditions)
    X = X[keep]
    y = pd.Categorical(file_list.loc[keep, "condition"].to_numpy(), categories=conditions).codes
print("X:", X.shape, "| y:", y.shape)
print(pd.Series(y).value_counts().sort_index().to_string())

X: (409, 110) | y: (409,)
0     79
1    276
2     28
3     15
4     11


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

binary = np.isin(y, [0, 1])
Xb, yb = X[binary], y[binary]
yb = (yb == 1).astype(int)

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    Xb, yb, test_size=0.3, stratify=yb, random_state=0
)

model = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(solver="saga", l1_ratio=0.5, max_iter=5000, random_state=0)),
])
model.fit(Xb_train, yb_train)

print("PD vs HC accuracy:", accuracy_score(yb_test, model.predict(Xb_test)))
print(classification_report(yb_test, model.predict(Xb_test), target_names=["HC", "Parkinson's"]))

PD vs HC accuracy: 0.7476635514018691
              precision    recall  f1-score   support

          HC       0.41      0.29      0.34        24
 Parkinson's       0.81      0.88      0.84        83

    accuracy                           0.75       107
   macro avg       0.61      0.59      0.59       107
weighted avg       0.72      0.75      0.73       107



In [5]:
target_names = ["HC", "Parkinson's", "Essential Tremor", "Atypical PD", "MS"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=0
)

multi = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(solver="saga", l1_ratio=0.5, max_iter=5000, random_state=0)),
])
multi.fit(X_train, y_train)

print("5-class accuracy:", accuracy_score(y_test, multi.predict(X_test)))
print(classification_report(y_test, multi.predict(X_test), target_names=target_names))

5-class accuracy: 0.6178861788617886
                  precision    recall  f1-score   support

              HC       0.33      0.33      0.33        24
     Parkinson's       0.73      0.77      0.75        83
Essential Tremor       0.50      0.50      0.50         8
     Atypical PD       0.00      0.00      0.00         5
              MS       0.00      0.00      0.00         3

        accuracy                           0.62       123
       macro avg       0.31      0.32      0.32       123
    weighted avg       0.59      0.62      0.60       123

